# Chapter 1: Introduction and Motivation

*This chapter sets the stage for the book by discussing the motivation for using quantum computing in many-body physics. We explain the exponential growth of Hilbert space in interacting many-body systems and why classical simulation methods become infeasible as system size increases. The historical context of quantum simulation is presented, from Feynman's original idea to modern experiments with near-term quantum devices. We also highlight recent initiatives (such as TALENT quantum computing courses and new quantum hardware) that have renewed interest in this field. The chapter concludes with an overview of the book's structure and key goals.*


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\matr}[1]{\boldsymbol{#1}}
\newcommand{\bfv}[1]{\boldsymbol{#1}}
\renewcommand{\vec}[1]{\boldsymbol{#1}}
\newcommand{\uvec}[1]{\boldsymbol{\hat{#1}}}
\newcommand{\op}[1]{\widehat{#1}}
\newcommand{\Op}[1]{\boldsymbol{\widehat{#1}}}
\newcommand{\OP}[1]{\boldsymbol{\widehat{#1}}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\real}[1]{\mathrm{Re}\left\{#1\right\}}
\newcommand{\ud}[1]{d#1}
\newcommand{\udd}[1]{d^2#1}
\newcommand{\mean}[1]{\langle #1 \rangle}
\newcommand{\overlap}[2]{\langle #1 | #2 \rangle}
\newcommand{\bracket}[3]{\langle #1 | #2 | #3 \rangle}
\newcommand{\projection}[2]{| #1 \rangle \langle #2 |}
\newcommand{\brab}[1]{\big\langle #1 \big|}
\newcommand{\ketb}[1]{\big| #1 \big\rangle}
\newcommand{\bket}[1]{\vert #1\rangle}
\newcommand{\bbra}[1]{\langle #1\vert}
\newcommand{\ketbra}[2]{\vert #1\rangle\langle #2\vert}
\newcommand{\braket}[2]{\langle #1\vert #2\rangle}
$$

In [ ]:
# Put every BookPrograms/chapter*/ and BookPrograms/common/ on sys.path
import sys, pathlib
root = pathlib.Path.cwd().resolve()
while not (root / 'BookPrograms').is_dir() and root.parent != root:
    root = root.parent
sys.path[:0] = [str(p) for p in (root / 'BookPrograms').iterdir() if p.is_dir()]
import numpy as np


## The problem

Almost everything interesting in physics involves many interacting
particles. The electrons of a molecule, the nucleons of a nucleus, the
spins of a magnet, the quarks and gluons of a hadron, the atoms in an
optical lattice: in every case the quantities one wants, a binding
energy, a spectrum, a response function, a phase diagram, follow in
principle from a Hamiltonian that can be written on one line and a
Schr\"odinger equation whose solution is, in practice, out of reach.
The reason is not that the equation is complicated. It is that its
solution is large. A system of $N$ fermions distributed over $M$
single-particle states has a Hilbert space of dimension $\binom{M}{N}$,
and a chain of $n$ spins one of dimension $2^n$; doubling the system
squares the dimension, and no reorganization of the calculation removes
that. Chapter 3 gives the numbers. Forty
single-particle states at half filling is a space of $10^{11}$
dimensions, the limit of what an iterative eigensolver on a large
parallel machine can handle; eighty states is $10^{23}$, more than the
number of bytes of storage on Earth; a $6\times6$ Hubbard cluster at
half filling has $8\times10^{19}$ states, and $^{16}$O in a modest
model space of forty orbitals per nucleon species has $6\times10^{15}$
determinants. This is the *exponential wall*, and it stands
between us and the exact solution of every problem in this book.

Many-body theory is the art of not solving the Schr\"odinger equation
exactly. Mean-field theory replaces the interacting problem by a
product state in a self-consistent potential and captures, at
polynomial cost, the bulk of the energy and none of the correlations;
perturbation theory and coupled-cluster theory build the correlations
back in with a polynomial number of parameters, and are exact in a
limit that is rarely reached; configuration interaction truncates the
space and converges slowly; quantum Monte Carlo samples the space
stochastically and, for fermions, pays for it with a sign problem that
grows exponentially in the particle number or the inverse temperature;
tensor networks compress the state and succeed exactly when the
entanglement is small, which in one dimension is often and in more is
not. Each of these has produced results of lasting value, and each has
a regime in which it fails, and the regimes of failure overlap: strongly
correlated fermions in two or three dimensions, real-time dynamics far
from equilibrium, and the spectra of systems with many competing scales
are hard for all of them at once. The exponential wall is not an
inconvenience of the current methods but a property of the problem, and
the question this book asks is what a computer whose own state space
grows exponentially can do about it.


## Feynman's idea and what became of it

The idea is older than the machines. In 1981, at a conference on the
physics of computation, Richard Feynman asked whether a classical
computer could simulate quantum mechanics efficiently, answered that in
general it could not, since the number of amplitudes to be tracked
grows exponentially with the size of the system, and proposed the
obvious remedy: build a computer out of quantum-mechanical elements,
whose state space grows in the same way, and let one quantum system
imitate another [Feynman1982]. The proposal was a thought
experiment for a decade. Deutsch formalized the notion of a universal
quantum computer, Shor's factoring algorithm of 1994 showed that such a
machine could do something classical computers apparently cannot, and
in 1996 Lloyd showed that Feynman's simulator was universal in the
sense that matters here: any local Hamiltonian can be simulated by a
sequence of short evolutions under its pieces, the Trotter formula of
Chapter 3, with a cost that grows polynomially in the
size of the system and the length of the evolution [Lloyd1996].
Abrams and Lloyd then showed how to turn simulated dynamics into
eigenvalues, by phase estimation [AbramsLloyd1999], and
Aspuru-Guzik and collaborators worked out in 2005 what it would cost
to compute the ground-state energy of a molecule that way
[AspuruGuzik2005]. By then the algorithmic side of quantum
simulation was, in outline, complete: map the fermions to qubits,
prepare a trial state, evolve it under the Hamiltonian, read off the
energy with a phase register. Chapters 4
to 6 of this book follow that outline.

The machines caught up slowly and then quickly. The first
demonstrations, on two or three qubits of nuclear spins, trapped ions
or superconducting circuits, computed the energy of the hydrogen
molecule or simulated a two-site model, and their point was that the
outline could be executed at all. Between 2014 and 2020 a second
family of algorithms appeared, adapted to machines with tens of noisy
qubits and no error correction: the variational quantum eigensolver of
Peruzzo and collaborators [Peruzzo2014], which replaces the deep
coherent circuit of phase estimation by a shallow parametrized one and
a classical optimizer, and its relatives, which are the subject of
Chapter 7. Preskill named the era, noisy intermediate-scale
quantum computing [Preskill2018], and predicted with some accuracy
what it would and would not deliver. For nuclear physics the entry
point was the calculation of the deuteron binding energy on a
cloud-accessible superconducting processor in 2018 [Dumitrescu2018],
a two- and three-qubit calculation whose importance was not the number
but the demonstration that the machinery of effective field theory,
the Jordan--Wigner mapping and a variational circuit could be put
together end to end. Since then the field has computed the spectra of
small nuclei and lattice gauge theories, the dynamics of spin models on
a hundred or more qubits at depths where classical simulation becomes
laborious [Kim2023], and, most consequentially, has crossed the
threshold at which a logical qubit encoded in many physical ones is
better than its constituents [Acharya2025,Bluvstein2024]. That
last result is the one that makes the fault-tolerant algorithms of
Chapters 6 and 8 a matter of engineering
rather than of principle, and it is the reason this book treats them
on an equal footing with the near-term ones.

The community around these questions has organized itself
accordingly. The long-range planning exercises of nuclear physics in
the United States and Europe now list quantum information science among
their priorities [Beck2023]; the TALENT schools (Training in
Advanced Low Energy Nuclear Theory), from which parts of this book
descend, have in recent years run courses on quantum computing and quantum
machine learning for nuclear physics; and university curricula that a decade ago offered quantum
computing as an elective in the physics of information now offer it
as a tool of many-body theory. This book is written for the graduate
student or researcher who arrives from the many-body side, knows what a
Slater determinant and a coupled-cluster amplitude are, and wants to
know, concretely and with code, what a quantum computer does with
them.


## What a quantum computer is, and is not, for a many-body physicist

It is worth being precise at the outset about the kind of advantage
that is on offer, because the subject attracts both overstatement and
its reaction. A register of $n$ qubits holds a state of $2^n$ complex
amplitudes, and a circuit of polynomial depth transforms all of them
at once. That is the source of every advantage in this book, and it is
a source with three restrictions attached. First, the amplitudes cannot
be read: a measurement returns $n$ bits drawn from the distribution of
the squared amplitudes and destroys the state, so that every quantity
one wants must be arranged to appear as the statistics of a few bits
over many repetitions, and Chapters 4 and 7
show what that costs. Second, the state must be prepared: an
eigenstate of an interacting Hamiltonian is no more available on a
quantum computer than on a classical one, and the algorithms that
extract its energy exactly, phase estimation and its descendants, need
a trial state whose overlap with it is not exponentially small. That
overlap is the quantity on which the promise of exponential advantage
for ground-state problems hinges, and it is not known to be large in
general [Lee2023]. Third, the machines are noisy, and the depth
of a circuit that can be executed before the noise overwhelms the
signal is, on current hardware, of the order of a few hundred to a few
thousand two-qubit gates, a budget that the algorithms of
Chapter 6 exceed by orders of magnitude and the algorithms
of Chapter 7 were designed to respect. Quantum error
correction, Chapter 9, removes the third restriction at a
large constant cost in qubits, and does nothing about the first two.

What survives these restrictions is real. Real-time dynamics is the
clearest case: a classical computer must store the state, and a
quantum computer need only propagate it, so that the evolution of an
interacting system for a time $t$ costs resources polynomial in $t$ and
in the size of the system, which no classical method achieves in
general; this is Lloyd's theorem, and it is the content of
Chapter 5 and the reason the lattice gauge theories
of Chapter 10, whose real-time and finite-density physics
is closed to lattice Monte Carlo by the sign problem, are the
application with the strongest motivation. Eigenvalues are the second
case, with the overlap caveat above: given a state that is a fair
approximation to an eigenstate, phase estimation refines its energy to
a precision exponential in the number of qubits spent on it, a
guarantee no variational method has. And the variational algorithms,
whose guarantees are weaker, are the ones that run today and the ones
through which most of what is known empirically about these machines
has been learned. The honest summary is that quantum computers move
the difficulty of the many-body problem rather than remove it, from
the size of the state to its preparation and measurement, and the
book is organized around seeing exactly where it goes.


## The approach of this book

Three decisions shape the book. The first is to follow a small number
of model Hamiltonians all the way through. Chapter 3
introduces four of them, the Lipkin model, the pairing model, the
Hubbard model and the Heisenberg model, each a caricature of a class of
physical systems, each exactly solvable at small size by symmetry, and
each the standard test case for some family of methods. They are then
solved classically by exact diagonalization, Hartree--Fock and coupled
cluster in Chapter 3, mapped to qubits in
Chapter 4, time-evolved in Chapter 5,
handed to phase estimation in Chapter 6 and to the
variational algorithms in Chapter 7, and filtered in
Chapter 8, at the same sizes and couplings throughout, so
that the tables of different chapters can be laid side by side. Table 1.1
collects the four. The comparison between methods is more instructive
than any one of them, and the thread of one problem through the whole
book is what makes the comparison possible.

| model | stands for | symmetry used | size in the book |
|---|---|---|---|
| Lipkin | collective nuclear motion | quasispin $J$ | $N=4$, four qubits |
| pairing | superconductivity, nuclear pairing | seniority | $L=4$, two pairs, four qubits |
| Hubbard | correlated electrons, cold atoms | particle number, $S_z$ | four-site ring, eight qubits |
| Heisenberg | quantum magnetism, spin chains | $S^z$, total spin | four-site ring, four qubits |

*Table 1.1: The four model Hamiltonians followed through the book, the
physics each stands for, the symmetry that makes small instances
exactly solvable, and the size used in the calculations of
Chapters 3--7.*

The second decision is that every number in the book is produced by a
program the reader can run. The programs are written in Python on
`numpy` and `matplotlib` alone, and they are not calls to
a quantum-computing framework: the state-vector simulator, the gates,
the circuits, the Pauli sums, the Jordan--Wigner mapping, the Trotter
steps, the phase-estimation register and the variational loop are all
written out, in a few hundred lines each, so that nothing is hidden
and everything can be modified. Each chapter ends with a list of its
programs and is accompanied by a Jupyter notebook, generated from the
chapter text, that executes every listing and every program in order.
The frameworks that run on real hardware, Qiskit, Cirq, PennyLane and
their relatives, are introduced in Appendix A once the reader knows
what they do, and the translation from the book's simulator to any of
them is mechanical. A reader who works through the notebooks will
have, at the end, a complete and understood toolkit; a reader who only
reads will still find every claim backed by a table whose provenance
is stated.

The third decision is a matter of tone. Quantum computing for
many-body physics is a young field in which the gap between what has
been proposed and what has been demonstrated is wide, and the book
tries to keep the two apart. Where an algorithm has a theorem behind
it, the theorem is stated and, when it is short, proved; where a
method is heuristic, the book says so and shows, on the models, both
where it works and where it fails, since the failures of variational
circuits and of trial states are as instructive as their successes
and considerably less advertised. Where the answer to a question is not
known, whether the overlap problem is benign for nuclei, whether
variational circuits can be trained at scale, whether any of this beats
the best classical methods on a problem anyone cares about, the book
says that too.


## A guide to the chapters

The book has three parts, though they are not marked as such.
Chapters 2 to 4 are
foundations. Chapter 2 sets out the gate model
of quantum computation in the language of linear algebra that a
physicist already speaks: qubits, the Bloch sphere, Pauli matrices,
tensor products, measurement, density matrices and decoherence,
entanglement, the elementary gates and the circuits built from them,
with superdense coding and teleportation as first examples of what
entanglement buys and a variational calculation on a two-level system
as a first glimpse of Chapter 7. Chapter 3
is the many-body side of the same foundations: second quantization,
the four models, and the three classical methods, exact
diagonalization, Hartree--Fock and coupled cluster, against which every
quantum result in the book is measured, ending with the unitary
coupled-cluster ansatz whose evaluation is exactly the thing a
classical computer cannot do and a quantum computer can.
Chapter 4 joins the two: the Jordan--Wigner
transformation turns the second-quantized Hamiltonians into sums of
Pauli strings, symmetries reduce the number of qubits and strings,
and three circuits follow, the exponential of a Pauli string, the
Trotter step and the measurement of an energy, which are the building
blocks of everything after.

Chapters 5 to 8 are algorithms.
Chapter 5 is Hamiltonian simulation: the
Trotter--Suzuki decomposition of $e^{-i\hat{H}t}$ and its error, the
higher-order formulas, the modern alternatives of qubitization and
quantum signal processing, real-time dynamics of the lattice models,
and the non-unitary evolutions, imaginary time in particular, that can
be embedded in a unitary at the price of a probability of success.
Chapter 6 is the quantum Fourier transform and phase
estimation, the algorithm that turns simulated dynamics into
eigenvalues, applied to the four models with a careful account of the
two errors, the finite register and the Trotterized evolution, and of
the trial state; the Harrow--Hassidim--Lloyd algorithm for linear
systems is presented there as an application of the same register to
the resolvent of a many-body Hamiltonian. Chapter 7 is
the variational family: the variational quantum eigensolver, its
ansätze, its measurement cost, its gradients and its failure modes,
applied to the same four models and compared with phase estimation;
the adaptive version that builds its own circuit; and the quantum
approximate optimization algorithm, with a glance at quantum machine
learning. Chapter 8 is the rodeo algorithm and its
relatives, which use controlled evolutions and measurements to filter
an eigenstate out of a trial state, and which sit between phase
estimation and the variational methods in what they require and what
they guarantee. Because every filter needs a trial state with a large
overlap, the chapter begins with resolution refinement, which carries
an eigenstate adiabatically from a coarse model space to a fine one; it
runs the combination on all four models, and it ends with a gate-level
depth budget that collects the bottlenecks met along the way.

Chapters 9 to 11 are about the machines
and the frontier. Chapter 9 is quantum error correction:
the error models, the repetition and Shor codes, syndrome measurement,
the surface code and the threshold theorem, and the error-mitigation
techniques that stand in for error correction on present hardware.
Chapter 10 is a case study in the direction the field is
heading, the simulation of a lattice gauge theory, the Schwinger model
in one spatial dimension, from the Kogut--Susskind Hamiltonian through
the mapping of matter and gauge fields to qubits to circuits for its
dynamics and observables, with results from exact diagonalization and
from quantum hardware and a discussion of what changes in more
dimensions and with non-Abelian gauge groups. Chapter 11
closes the book with an assessment of where the hardware stands, of
what the analog simulators, the annealers and the hybrid methods add
to the gate-model picture, and of the open problems. Appendix A
collects the software tools, and Appendix B the mathematical
background. Table 1.2 gives the map in one place.

|  | chapter | question | tool |
|---|---|---|---|
| 2 | Fundamentals | what is a qubit, a gate, a measurement? | state-vector simulator, circuits |
| 3 | Many-body primer | what are we trying to compute? | second quantization, the four models, FCI, HF, CC |
| 4 | Mapping to qubits | how does a Hamiltonian reach the machine? | Jordan--Wigner, Pauli sums, symmetry encodings |
| 5 | Hamiltonian simulation | how is $e^{-i\hat{H}t}$ built? | Trotter--Suzuki, higher order, qubitization |
| 6 | QFT and phase estimation | how is an eigenvalue read out? | QFT, QPE, HHL |
| 7 | Variational algorithms | what runs on noisy hardware? | VQE, ADAPT-VQE, QAOA |
| 8 | Rodeo and filtering | how is an eigenstate isolated? | rodeo algorithm, spectral filters |
| 9 | Error correction | how are errors removed? | codes, syndromes, surface code, mitigation |
| 10 | Lattice gauge theories | where is this heading? | Schwinger model on qubits |
| 11 | Outlook | what is open? | hardware, analog simulators, perspectives |

*Table 1.2: The chapters, the question each answers, and the tool it
introduces.*


## How to read this book

The prerequisites are a first course in quantum mechanics, linear
algebra at the level of eigenvalue problems and tensor products, and
some acquaintance with second quantization, all of which
Chapters 2 and 3 and
Appendix B review at the pace of a reminder rather than a first
encounter. No prior knowledge of quantum computing is assumed. A
reader who has it can begin at Chapter 3; a reader
who comes from quantum information rather than many-body physics
should read Chapter 3 with care, since the models and
the classical methods it introduces are what every later chapter
computes and compares against. The chapters are meant to be read in
order through Chapter 7, after which
Chapters 8, 9 and 10 can be
taken in any order.

A few conventions hold throughout and are worth fixing now, because
the literature does not agree on them. Qubit $0$ is the leftmost
factor of a tensor product and the most significant bit of a
computational basis state, so that $\vert q_0q_1\cdots q_{n-1}\rangle$
is read as a binary number from the left; in the occupation-number
encodings, $\vert 1\rangle$ is an occupied mode. Circuits are read
from left to right, operators from right to left. Rotations are
$R_a(\theta)=e^{-i\theta\sigma_a/2}$, so that a full turn is $\theta=2\pi$
and changes the sign of the state. Hamiltonians carry a hat, $\hat{H}$,
and their qubit images are written as Pauli sums $\sum_\alpha h_\alpha P_\alpha$
with strings $P_\alpha$ of the matrices $I$, $X$, $Y$, $Z$. Energies
are quoted in the units of the model, with $\hbar=1$ throughout. The
exercises at the end of each chapter range from short derivations to
numerical projects that extend the chapter's programs, and the
programs, which are listed at the end of each chapter and collected in
the accompanying repository, are as much a part of the book as the
text.
